CELL 1: Settings and Helpers

In [ ]:
from pathlib import Path
import time
import numpy as np, pandas as pd, geopandas as gpd, xarray as xr, rasterio
import rioxarray  # noqa: F401  (adds .rio to xarray, for saving GeoTIFFs)
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, to_hex
from matplotlib.patches import Patch, Rectangle
from rasterio.windows import from_bounds, bounds as win_bounds
from malkus import TrackSet, RegularGrid, compute_winds, SurfaceRoughness, downscale_winds
from malkus.wind.downscale.roughness import roughness_from_land_cover, power_law_scale_factors
from scipy.ndimage import gaussian_filter

# --- files ---
ROOT = Path("/home/nerskine/malkus")
TRACKS = ROOT / "data/in/tracks/lesser-antilles"
GWA4_URL = "/vsicurl/https://api.globalwindatlas.info/cogs/GWA4_ESA_WorldCover_2021_50m.tif"   # DTU Global Wind Atlas v4
LC  = ROOT / "data/in/land_cover/gwa4/GWA4_landcover_50m_StVincent_to_Martinique.tif"           # downloaded in Cell 2a
MAP = ROOT / "data/in/land_cover/gwa4_to_surface_roughness.csv"                                 # written in Cell 4
LC_WORLDCOVER = ROOT / "data/in/land_cover/esa_worldcover_2021/ESA_WorldCover_10m_2021_v200_N12W063_Map.tif"   # raw WorldCover
MAP_WORLDCOVER_COPIED = ROOT / "data/in/land_cover/worldcover_to_surface_roughness.csv"         # earlier copied table (comparison)
LC_GLOBCOVER  = ROOT / "data/in/land_cover/glob_cover_2009/GLOBCOVER_L4_200901_200912_V2.3.tif" # old map (comparison)
MAP_GLOBCOVER = ROOT / "data/in/land_cover/land_cover_to_surface_roughness.csv"
OUT = ROOT / "data/out/st_lucia"
CROP_OUT = ROOT / "data/in/land_cover/esa_worldcover_2021/ESA_WorldCover_2021_StVincent_to_Martinique.tif"

# --- land types (WorldCover classes + GWA4 tree-height classes) ---
WC_NAMES = {0: "No data / sea", 10: "Tree cover, height unknown", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
            50: "Built-up", 60: "Bare / sparse vegetation", 70: "Snow and ice", 80: "Permanent water",
            90: "Herbaceous wetland", 95: "Mangroves", 100: "Moss and lichen",
            101: "Trees under 2 m", 116: "Trees over 30 m",
            **{k: f"Trees ~{3 + 2 * (k - 102)} m" for k in range(102, 116)}}
WATER_CLASSES = [0, 80]                    # treated as sea when drawing coastlines and counting land
TREE_CLASSES = list(range(101, 117))       # GWA4 tree-height classes
COAST_STEP = 1                             # GWA4 pixels are 50 m: read every pixel for the coastline

# land-cover colours (official WorldCover colours; greens from short to tall trees) — used by Cells 2-pre and 2
WC_COLOURS = {0: "#ffffff", 10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff", 50: "#fa0000",
              60: "#b4b4b4", 70: "#f0f0f0", 80: "#0064c8", 90: "#0096a0", 95: "#00cf75", 100: "#fae6a0",
              **{k: to_hex(plt.get_cmap("Greens")(0.3 + 0.7 * (k - 101) / 15)) for k in TREE_CLASSES}}
codes = list(WC_COLOURS)
lut = np.zeros(256, dtype=int)             # class code -> 0, 1, 2 ... so each code gets exactly one colour
for n, k in enumerate(codes):
    lut[k] = n
wc_cmap = ListedColormap([WC_COLOURS[k] for k in codes])
wc_norm = BoundaryNorm(np.arange(len(codes) + 1) - 0.5, len(codes))

# --- datasets (one present-day catalogue per model family), and the one storm shown from each ---
CATALOGUES = {
    "CHAZ (2010, UKESM1)":   "source-chaz_epoch-2010_scenario-none_gcm-UKESM1",
    "Emanuel (2005, UKMO6)": "source-emanuel_epoch-2005_scenario-none_gcm-UKMO6",
    "IRIS (2000)":           "source-iris_epoch-2000_scenario-none_gcm-none",
    "STORM (2000)":          "source-storm_epoch-2000_scenario-none_gcm-none",   # STORM_DATA_IBTRACS_NA, 1-min winds
}
SHOWCASE = {
    "CHAZ (2010, UKESM1)":   "S000Y0921N014",
    "Emanuel (2005, UKMO6)": "S000Y0194N065",      # 61.6 m/s, passes 35 km from Castries
    "IRIS (2000)":           "NA_0_370_4",
    "STORM (2000)":          "NA_0_966_0",
}

# --- areas, grids and analysis choices ---
BBOX = (-61.3, 13.6, -60.7, 14.2)          # area winds are calculated for (west, south, east, north)
VIEW = (-61.10, 13.69, -60.85, 14.13)      # area shown on the maps: St Lucia, filling the panel
CROP = (-61.6, 12.95, -60.6, 15.0)         # St Vincent to Martinique, with sea (shared folder + GWA4 download)
RESOLUTION = 0.01                          # all-storm grid (~1.1 km)
FINE = 1 / 1200                            # showcase + roughness grid (~90 m = 2 x 2 GWA4 pixels, aligned)
CASTRIES = dict(lat=14.01, lon=-60.99)
RETURN_PERIODS = [10, 25, 50, 100, 250, 500]
HURRICANE_FORCE = 33.0                     # m/s
SHOW_HURRICANE_LINE = False                # True = draw the dashed 33 m/s outline on footprint maps
MAX_CENTRAL_PRESSURE_HPA = 1013.0          # points at/above the 1014.1 hPa background crash the wind formula
SPACING = 0.2                              # eye positions every 0.2 x storm size (Cell 6 checks this)
N_WORKERS = 4

grid = RegularGrid.from_bbox(BBOX, RESOLUTION)
fine_grid = RegularGrid.from_bbox(BBOX, FINE)
roughness = SurfaceRoughness(land_cover_path=LC, mapping_path=MAP)

# --- look ---
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": "0.65", "axes.linewidth": 0.6,
    "xtick.color": "0.45", "ytick.color": "0.45", "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
})
WIND_CMAP, RISK_CMAP, KEPT_CMAP, CUT_CMAP = "magma_r", "Reds", "RdYlBu_r", "Blues_r"
KEPT_RANGE = (65, 95)                      # colour range for "% of wind kept" (tall trees keep ~67%)
NOTE = dict(fontsize=8, va="bottom", bbox=dict(fc="white", ec="none", alpha=0.85, pad=2))
MAP_H = 5.0                                                                  # height of one map panel (inches)
MAP_W = MAP_H * (VIEW[2] - VIEW[0]) / (VIEW[3] - VIEW[1]) + 0.7             # width that fits the island's shape

# --- helpers ---
def map_figure(nrows, ncols, extra=1.3, scale=1.0, **kw):
    """A figure whose map panels are shaped like St Lucia, so the island fills each panel."""
    return plt.subplots(nrows, ncols, figsize=(MAP_W * scale * ncols + extra, MAP_H * scale * nrows + 0.6),
                        squeeze=False, layout="constrained", **kw)

_coast = {}
def coastline(ax, bbox=VIEW, step=COAST_STEP):
    """Draw the coastline from the land-cover water classes and zoom to bbox."""
    if bbox not in _coast:
        with rasterio.open(LC) as src:
            w = from_bounds(*bbox, src.transform).round_offsets().round_lengths()
            _coast[bbox] = (~np.isin(src.read(1, window=w)[::step, ::step], WATER_CLASSES), win_bounds(w, src.transform))
    is_land, b = _coast[bbox]
    ax.contour(is_land, levels=[0.5], colors="0.15", linewidths=0.7, extent=(b[0], b[2], b[1], b[3]), origin="upper")
    ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3]); ax.set_aspect("equal"); ax.tick_params(length=2)

def on_grid(values, g):
    """Wrap a numpy array on grid g as a lat/lon DataArray."""
    return xr.DataArray(values, dims=("lat", "lon"), coords=dict(lat=g.lats, lon=g.lons))

def in_view(da):
    return da.sel(lon=slice(VIEW[0], VIEW[2]), lat=slice(VIEW[1], VIEW[3]))

def limits(fields, step=1):
    """Colour limits fitted to what is visible on the zoomed maps."""
    v = np.concatenate([in_view(f).values.ravel() for f in fields])
    return step * np.floor(np.nanmin(v) / step), step * np.ceil(np.nanmax(v) / step)

def show(ax, da, cmap, vmin, vmax, title, **kw):
    """One map panel: field, coastline, zoom to the island, a star for Castries, tidy axis labels."""
    lon, lat = da["lon"].values, da["lat"].values
    dx, dy = lon[1] - lon[0], lat[1] - lat[0]
    im = ax.imshow(da.values, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest",
                   extent=(lon[0] - dx / 2, lon[-1] + dx / 2, lat[0] - dy / 2, lat[-1] + dy / 2), **kw)
    coastline(ax)
    ax.plot(CASTRIES["lon"], CASTRIES["lat"], marker="*", ms=9, mfc="white", mec="black", mew=0.7, ls="none")
    ax.locator_params(axis="x", nbins=3); ax.locator_params(axis="y", nbins=5)
    ax.set_title(title)
    return im

def hurricane_line(ax, da, smooth_m=300):
    """Dashed white outline of where wind exceeded hurricane force.
    On fine grids the field is lightly smoothed first (smooth_m metres), so the line shows the
    general boundary instead of every pixel-scale flip caused by patchy land cover."""
    if not SHOW_HURRICANE_LINE:
        return
    vals = da.values
    cell_m = abs(float(da["lon"][1] - da["lon"][0])) * 111_000
    if smooth_m and smooth_m / cell_m >= 0.5:                       # only smooths fine grids (e.g. 90 m), not 1.1 km
        vals = gaussian_filter(vals, sigma=smooth_m / cell_m)
    if np.nanmax(vals) > HURRICANE_FORCE > np.nanmin(vals):
        ax.contour(da["lon"], da["lat"], vals, levels=[HURRICANE_FORCE], colors="white", linewidths=0.9, linestyles="--")

def at_castries(da):
    return float(da.sel(**CASTRIES, method="nearest"))

def showcase_track(label):
    """The chosen showcase storm of one dataset, as its own TrackSet."""
    c = catalogues[label]
    return TrackSet(c["ts"].tracks[c["ts"].tracks["track_id"] == SHOWCASE[label]], source=c["source"])

def annual_maxima(W, n_years):
    """Worst wind in each year; storm-free years count as 0 m/s (malkus's own function forgets them)."""
    a = W.groupby("year").max("event")
    if n_years > a.sizes["year"]:
        a = xr.concat([a, xr.zeros_like(a.isel(year=0)).expand_dims(year=[-1])], "year")
    return a

def rp_from_annual(a, n_years, periods):
    """1-in-N-year values for every grid cell from yearly worst winds.
    Return periods longer than the catalogue (N > n_years + 1) can't be estimated and are left blank (NaN)."""
    ranked = np.sort(a.values, axis=0)
    rp = (n_years + 1) / np.arange(len(ranked), 0, -1)
    flat = ranked.reshape(len(ranked), -1)
    out = np.stack([np.interp(periods, rp, flat[:, c]) for c in range(flat.shape[1])], axis=1)
    out[np.asarray(periods) > n_years + 1] = np.nan                  # e.g. Emanuel (~234 yrs): no 1-in-250 or 1-in-500
    return xr.DataArray(out.reshape(len(periods), *ranked.shape[1:]), dims=("return_period", "lat", "lon"),
                        coords=dict(return_period=periods, lat=a.lat, lon=a.lon))

def rp_maps(W, n_years, periods):
    """1-in-N-year wind for every grid cell, using the full catalogue length."""
    return rp_from_annual(annual_maxima(W, n_years), n_years, periods)

def chance_of_hurricane_force(res):
    """% chance per year of exceeding HURRICANE_FORCE, for every grid cell."""
    return {label: 100 * (annual_maxima(r["wind"], r["n_years"]) > HURRICANE_FORCE).sum("year") / r["n_years"]
            for label, r in res.items()}

def save_set(res, maps, tag):
    """Save one set (tag = 'open_terrain' or 'roughness'): footprints, storm list, worst-storm and return-period GeoTIFFs."""
    for label, r in res.items():
        name, w = label.split()[0].lower(), r["wind"]
        d = OUT / f"footprints_{tag}_{name}"; d.mkdir(parents=True, exist_ok=True)
        r["footprints"].data.to_zarr(d / "footprints.zarr", mode="w", consolidated=False)
        index = pd.DataFrame({"event": w.event.values, "year": w.year.values,
                              "peak_in_box_ms": w.max(["lat", "lon"]).values,
                              "wind_at_castries_ms": w.sel(**CASTRIES, method="nearest").values}
                             ).sort_values("peak_in_box_ms", ascending=False)
        index.to_csv(d / "storm_index.csv", index=False)
        tif = d / "worst_storms"; tif.mkdir(exist_ok=True)
        for event in index["event"].head(10):
            fp = w.sel(event=event).drop_vars("year").rename(lon="x", lat="y").sortby("y", ascending=False)
            fp.astype("float32").rio.write_crs("EPSG:4326").rio.to_raster(tif / f"{event}.tif")
        rp_dir = OUT / f"rp_maps_{tag}_{name}"; rp_dir.mkdir(parents=True, exist_ok=True)
        m = maps[label].rename(lon="x", lat="y").sortby("y", ascending=False).rio.write_crs("EPSG:4326")
        for T in RETURN_PERIODS:
            m.sel(return_period=T).astype("float32").rio.to_raster(rp_dir / f"rp_{T}yr.tif")
        print(f"{label} ({tag}) -> {d.name}/ and {rp_dir.name}/")

CHANCE_LEVELS = np.arange(HURRICANE_FORCE, 61.0, 0.5)     # open-terrain wind levels (m/s); 33 / lowest share kept is ~49

def to_fine(da):
    """Interpolate a smooth 1.1 km open-terrain field onto the 90 m grid."""
    return da.interp(lat=fine_grid.lats, lon=fine_grid.lons, kwargs={"fill_value": None})

def chance_above(annual, n_years, levels=CHANCE_LEVELS):
    """% of years whose worst open-terrain wind exceeds each level (1.1 km grid)."""
    return xr.concat([100 * (annual > v).sum("year") / n_years for v in levels],
                     dim=xr.DataArray(levels, dims="level", name="level"))

def fine_chance_with_roughness(annual, n_years, kept_fine):
    """% chance per year of hurricane-force wind WITH roughness, on the 90 m grid.
    A place keeping a share f of the wind reaches 33 m/s when the open-terrain wind exceeds 33 / f,
    so each 90 m cell looks up how often the open-terrain wind passes that level."""
    curves = to_fine(chance_above(annual, n_years)).values                  # (level, lat, lon) on 90 m
    need = HURRICANE_FORCE / kept_fine.values                               # open-terrain wind needed per cell
    pos = np.clip((need - CHANCE_LEVELS[0]) / (CHANCE_LEVELS[1] - CHANCE_LEVELS[0]), 0, len(CHANCE_LEVELS) - 1)
    i0 = np.floor(pos).astype(int); i1 = np.minimum(i0 + 1, len(CHANCE_LEVELS) - 1); w = pos - i0
    lo = np.take_along_axis(curves, i0[None], 0)[0]; hi = np.take_along_axis(curves, i1[None], 0)[0]
    return on_grid(lo * (1 - w) + hi * w, fine_grid)

def save_rp_geotiffs(md, folder_tag):
    """Save return-period maps (any grid) as GeoTIFFs, one folder per dataset."""
    for label, m in md.items():
        d = OUT / f"rp_maps_{folder_tag}_{label.split()[0].lower()}"; d.mkdir(parents=True, exist_ok=True)
        m = m.rename(lon="x", lat="y").sortby("y", ascending=False).rio.write_crs("EPSG:4326")
        for T in RETURN_PERIODS:
            m.sel(return_period=T).astype("float32").rio.to_raster(d / f"rp_{T}yr.tif")
        print(f"{label} -> {d.name}/")

def plot_ring_test(refs, test_fields, values, test_name, fmt):
    """Coffee cup ring test, one figure per dataset (Cell 7c).
    Top row: footprint at each test value, plus the reference. Bottom row: wind missed
    (reference minus test) on a log scale, so small errors still show."""
    for label in catalogues:
        ref = refs[label]
        ws = [test_fields[label, v] for v in values]
        lim = limits([ref] + ws, step=5)
        misses = [(ref - w).clip(min=0.05) for w in ws]                      # ≤0.05 m/s = no visible error
        miss_max = max(float(m.max()) for m in misses)
        n = len(values) + 1
        fig, axes = plt.subplots(2, n, figsize=(2.4 * n + 1.2, 5.8), layout="constrained", squeeze=False)
        for col, (v, w, miss) in enumerate(zip(values, ws, misses)):
            im_w = show(axes[0, col], w, WIND_CMAP, *lim, f"{test_name} {fmt(v)}")
            im_e = show(axes[1, col], miss, "Purples", 0.05, miss_max, f"worst miss {float((ref - w).max()):.2f} m/s",
                        norm="log")
        show(axes[0, -1], ref, WIND_CMAP, *lim, "reference\nspacing 0.05")
        axes[1, -1].axis("off")
        for ax in axes.flat:
            coastline(ax, bbox=BBOX)                                         # whole calculation box: rings reach the sea
            ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im_w, ax=axes[0], shrink=0.85, label="Maximum wind, open terrain (m/s)")
        fig.colorbar(im_e, ax=axes[1, :-1], shrink=0.85, label="Wind missed vs reference (m/s, log scale)")
        fig.suptitle(f"{label} · {SHOWCASE[label]} — coffee cup rings by {test_name}", fontweight="bold", x=0.01, ha="left")
        plt.show()


CELL 2A: Downloading DTU's land cover for the region

In [ ]:
LC.parent.mkdir(parents=True, exist_ok=True)
if LC.exists():
    print("Already downloaded:", LC)
else:
    with rasterio.open(GWA4_URL) as src:
        win = from_bounds(*CROP, src.transform).round_offsets().round_lengths()
        data = src.read(1, window=win)
        profile = src.profile.copy()
        profile.update(driver="GTiff", height=data.shape[0], width=data.shape[1], transform=src.window_transform(win),
                       compress="deflate", tiled=True, blockxsize=512, blockysize=512)
    with rasterio.open(LC, "w", **profile) as dst:
        dst.write(data, 1)
    print(f"Saved {LC}\n{data.shape[1]} x {data.shape[0]} pixels at 50 m, classes: {sorted(int(i) for i in np.unique(data))}")

CELL 2B: ESA WORLD COVER PLOT ONLY

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm, to_hex
from matplotlib.patches import Patch, Rectangle

WC_COLOURS = {0: "#ffffff", 10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff", 50: "#fa0000",
              60: "#b4b4b4", 70: "#f0f0f0", 80: "#0064c8", 90: "#0096a0", 95: "#00cf75", 100: "#fae6a0",
              **{k: to_hex(plt.get_cmap("Greens")(0.3 + 0.7 * (k - 101) / 15)) for k in TREE_CLASSES}}  # short -> tall trees
codes = list(WC_COLOURS)
lut = np.zeros(256, dtype=int)
for n, k in enumerate(codes):
    lut[k] = n
wc_cmap = ListedColormap([WC_COLOURS[k] for k in codes])
wc_norm = BoundaryNorm(np.arange(len(codes) + 1) - 0.5, len(codes))

with rasterio.open(LC_WORLDCOVER) as src:
    print("Map projection:", src.crs, "| pixel size:", src.res, "degrees  (~", round(src.res[0] * 111_000), "m)")
    rwin = from_bounds(*CROP, src.transform).round_offsets().round_lengths()
    esa_region = src.read(1, window=rwin, out_shape=(int(rwin.height) // 20, int(rwin.width) // 20))  # every 20th pixel: plenty for a small panel
    esa_region_bounds = win_bounds(rwin, src.transform)
    vwin = from_bounds(*VIEW, src.transform).round_offsets().round_lengths()
    esa_view = src.read(1, window=vwin)                                                               # full 10 m detail
    esa_view_bounds = win_bounds(vwin, src.transform)

ids, counts = np.unique(esa_view, return_counts=True)
esa_share = {int(i): 100 * c / esa_view.size for i, c in zip(ids, counts)}
esa_land = esa_view[~np.isin(esa_view, WATER_CLASSES)]
print(f"Trees: {100 * (esa_land == 10).mean():.1f}% of St Lucia's land (one class — no heights)")
print("Land types in the zoomed St Lucia view (% of view):")
for i, pct in sorted(esa_share.items(), key=lambda kv: -kv[1]):
    print(f"  {i:>3}  {WC_NAMES.get(i, '?? unknown'):<26} {pct:5.1f}%")

fig, (ax_reg, ax_view) = plt.subplots(1, 2, figsize=(12.5, 8), layout="constrained", gridspec_kw=dict(width_ratios=(0.5, 0.57)))
ax_reg.imshow(lut[esa_region], cmap=wc_cmap, norm=wc_norm, interpolation="nearest",
              extent=(esa_region_bounds[0], esa_region_bounds[2], esa_region_bounds[1], esa_region_bounds[3]))
ax_reg.add_patch(Rectangle(BBOX[:2], BBOX[2] - BBOX[0], BBOX[3] - BBOX[1], fill=False, ec="red", lw=1.5))
ax_reg.set_title("St Vincent to Martinique\nred = St Lucia calculation box")
ax_reg.set_aspect("equal"); ax_reg.locator_params(axis="x", nbins=3)

ax_view.imshow(lut[esa_view], cmap=wc_cmap, norm=wc_norm, interpolation="nearest",
               extent=(esa_view_bounds[0], esa_view_bounds[2], esa_view_bounds[1], esa_view_bounds[3]))
ax_view.plot(CASTRIES["lon"], CASTRIES["lat"], marker="*", ms=11, mfc="white", mec="black", mew=0.8, ls="none")
ax_view.set_xlim(VIEW[0], VIEW[2]); ax_view.set_ylim(VIEW[1], VIEW[3]); ax_view.set_aspect("equal")
ax_view.locator_params(axis="x", nbins=3)
ax_view.set_title("St Lucia — ESA WorldCover only (10 m)")
ax_view.legend(handles=[Patch(facecolor=WC_COLOURS.get(k, "#ffffff"), edgecolor="0.4", lw=0.4,
                              label=f"{k:>3} {WC_NAMES.get(k, '?? unknown')} ({pct:.1f}%)")
                        for k, pct in sorted(esa_share.items(), key=lambda kv: -kv[1])],
               loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=7, title="Land type (share of view)", frameon=False)
fig.suptitle("ESA WorldCover 2021 v200 — land cover without tree heights", fontweight="bold", x=0.01, ha="left")
plt.show()


CELL 2C: Now we want to see what the world cover actually looks like using the GWAv4 which presents the surface roughness table

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm, to_hex
from matplotlib.patches import Patch, Rectangle

WC_COLOURS = {0: "#ffffff", 10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff", 50: "#fa0000",
              60: "#b4b4b4", 70: "#f0f0f0", 80: "#0064c8", 90: "#0096a0", 95: "#00cf75", 100: "#fae6a0",
              **{k: to_hex(plt.get_cmap("Greens")(0.3 + 0.7 * (k - 101) / 15)) for k in TREE_CLASSES}}  # short -> tall trees

with rasterio.open(LC) as src:
    print("Map projection:", src.crs, "(malkus needs EPSG:4326)")
    print("Pixel size:", src.res, "degrees  (~", round(src.res[0] * 111_000), "m)")
    print("Covers:", src.bounds)
    region = src.read(1)
    region_bounds = src.bounds
    cls_box = src.read(1, window=from_bounds(*BBOX, src.transform).round_offsets().round_lengths())
    win = from_bounds(*VIEW, src.transform).round_offsets().round_lengths()
    cls_view = src.read(1, window=win)
    view = win_bounds(win, src.transform)

box_ids = [int(i) for i in np.unique(cls_box)]
unknown = [i for i in box_ids if i not in WC_NAMES]
print("\nClasses in the calculation box:", box_ids, "| unknown:", unknown or "none")

ids, counts = np.unique(cls_view, return_counts=True)
share = {int(i): 100 * c / cls_view.size for i, c in zip(ids, counts)}
land_px = cls_view[~np.isin(cls_view, WATER_CLASSES)]
print(f"\nTrees: {100 * np.isin(land_px, TREE_CLASSES).mean():.1f}% of St Lucia's land")
print("Land types in the zoomed St Lucia view (% of view):")
for i, pct in sorted(share.items(), key=lambda kv: -kv[1]):
    print(f"  {i:>3}  {WC_NAMES.get(i, '?? unknown'):<26} {pct:5.1f}%")

codes = list(WC_COLOURS)
lut = np.zeros(256, dtype=int)
for n, k in enumerate(codes):
    lut[k] = n
wc_cmap = ListedColormap([WC_COLOURS[k] for k in codes])
wc_norm = BoundaryNorm(np.arange(len(codes) + 1) - 0.5, len(codes))

fig, (ax_reg, ax_view) = plt.subplots(1, 2, figsize=(12.5, 8), layout="constrained", gridspec_kw=dict(width_ratios=(0.5, 0.57)))
ax_reg.imshow(lut[region[::4, ::4]], cmap=wc_cmap, norm=wc_norm, interpolation="nearest",
              extent=(region_bounds.left, region_bounds.right, region_bounds.bottom, region_bounds.top))
ax_reg.add_patch(Rectangle(BBOX[:2], BBOX[2] - BBOX[0], BBOX[3] - BBOX[1], fill=False, ec="red", lw=1.5))
ax_reg.set_title("St Vincent to Martinique\nred = St Lucia calculation box")
ax_reg.set_aspect("equal"); ax_reg.locator_params(axis="x", nbins=3)

ax_view.imshow(lut[cls_view], cmap=wc_cmap, norm=wc_norm, interpolation="nearest",
               extent=(view[0], view[2], view[1], view[3]))
ax_view.plot(CASTRIES["lon"], CASTRIES["lat"], marker="*", ms=11, mfc="white", mec="black", mew=0.8, ls="none")
ax_view.set_xlim(VIEW[0], VIEW[2]); ax_view.set_ylim(VIEW[1], VIEW[3]); ax_view.set_aspect("equal")
ax_view.locator_params(axis="x", nbins=3)
ax_view.set_title("St Lucia — land cover with tree heights (50 m)")
ax_view.legend(handles=[Patch(facecolor=WC_COLOURS.get(k, "#ffffff"), edgecolor="0.4", lw=0.4,
                              label=f"{k:>3} {WC_NAMES.get(k, '?? unknown')} ({pct:.1f}%)")
                        for k, pct in sorted(share.items(), key=lambda kv: -kv[1])],
               loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=7, title="Land type (share of view)", frameon=False)
fig.suptitle("Global Wind Atlas v4 land cover (ESA WorldCover 2021 + ETH tree heights)", fontweight="bold", x=0.01, ha="left")
plt.show()


Cell 3: crop the raw WorldCover (10 m) for the shared folder

In [ ]:
with rasterio.open(LC_WORLDCOVER) as src:
    win = from_bounds(*CROP, src.transform).round_offsets().round_lengths()
    data = src.read(1, window=win)
    profile = src.profile.copy()
    profile.update(height=data.shape[0], width=data.shape[1], transform=src.window_transform(win),
                   compress="deflate", tiled=True, blockxsize=512, blockysize=512)
    try:
        crop_colours = src.colormap(1)
    except ValueError:
        crop_colours = None
with rasterio.open(CROP_OUT, "w", **profile) as dst:
    dst.write(data, 1)
    if crop_colours:
        dst.write_colormap(1, crop_colours)
print(f"Saved {CROP_OUT}\n{data.shape[1]} x {data.shape[0]} pixels, {CROP_OUT.stat().st_size / 1e6:.0f} MB")
print(f"The GWA4 50 m crop for the same area (worth sharing too): {LC}")



Cell 4: write the GWA4 roughness table

In [ ]:
gwa4 = {0: ("No data / sea", 0.0002), 10: ("Tree cover, height unknown", 1.5), 20: ("Shrubland", 0.3),
        30: ("Grassland", 0.03), 40: ("Cropland", 0.1), 50: ("Built-up", 0.5), 60: ("Bare / sparse vegetation", 0.005),
        70: ("Snow and ice", 0.003), 80: ("Permanent water bodies", 0.0002), 90: ("Herbaceous wetland", 0.1),
        95: ("Mangroves", 0.7), 100: ("Moss and lichen", 0.01),
        101: ("Trees under 2 m", 0.3), 116: ("Trees over 30 m", 3.0),
        **{k: (f"Trees ~{3 + 2 * (k - 102)} m", round(0.1 * (3 + 2 * (k - 102)), 4)) for k in range(102, 116)}}
table = pd.DataFrame([{"category_name": name, "glob_cover_2009_id": k, "roughness_length_m": z0}
                      for k, (name, z0) in sorted(gwa4.items())])
header = '''# Land cover to surface roughness for the Global Wind Atlas v4 land-cover map (ESA WorldCover 2021 v200 + ETH tree heights).
# Source: Floors, R., Davis, N., Olsen, B. T., Badger, J., Hansen, B. O. (2025). Global Wind Atlas v4 (landcover).
#         Technical University of Denmark. Dataset. https://doi.org/10.11583/DTU.28955279.v2 (CC BY 4.0)
# Values as published in DTU's windkit library (landcover table "GWA4", GWA4_micro.json).
# Tree classes 101-116: z0 = 0.1 x tree height (2 m height bins from the ETH canopy height map, Lang et al. 2023).
# Water (80) and no data (0) set to 0.0002, DTU's stated sea roughness (the published table uses 0, which malkus cannot accept).
# DTU also gives a displacement height (2/3 x tree height) that malkus does not use; see Cell 16b for a tall-tree check.
'''
MAP.write_text(header + table.to_csv(index=False))
table["wind_kept_pct"] = (100 * power_law_scale_factors(table["roughness_length_m"].to_numpy())).round(1)
print(table.to_string(index=False))


CELL 5: Load the Storms

In [ ]:
catalogues = {}
for label, folder in CATALOGUES.items():
    source = folder.split("_")[0].split("-", 1)[1]                      # "chaz" / "iris" / "storm"
    raw = gpd.read_parquet(TRACKS / folder / "tracks.geoparquet")
    n_years = int(raw["year"].max()) + 1                                # full catalogue length
    bad = raw["min_pressure_hpa"] >= MAX_CENTRAL_PRESSURE_HPA
    ts = TrackSet(raw[~bad], source=source).filter_by_bbox(BBOX, search_radius_deg=3.0)
    if SHOWCASE[label] not in set(ts.track_ids):
        raise ValueError(f"{SHOWCASE[label]} is not among the {label} storms near St Lucia")
    catalogues[label] = dict(source=source, n_years=n_years, ts=ts, removed=int(bad.sum()))

print(pd.DataFrame({label: {"years simulated": c["n_years"],
                            "storms near St Lucia": len(c["ts"].track_ids),
                            "storms per year": round(len(c["ts"].track_ids) / c["n_years"], 2),
                            "track points removed": c["removed"],
                            "showcase storm": SHOWCASE[label]} for label, c in catalogues.items()}).to_string())


CELL 6: Coffee Cup Rings test

In [ ]:
TEST_SPACINGS = [0.7, 0.4, 0.3, 0.2, 0.1]
rows, refs, spacing_fields = [], {}, {}
for label in catalogues:
    one = showcase_track(label)
    ref = compute_winds(one, grid, interpolation_spacing_factor=0.05).data["max_wind_speed_ms"].isel(event=0)
    refs[label] = ref                                                   # "truth": very fine spacing
    for sp in TEST_SPACINGS:
        t0 = time.time()
        w = compute_winds(one, grid, interpolation_spacing_factor=sp).data["max_wind_speed_ms"].isel(event=0)
        spacing_fields[label, sp] = w                                   # kept for Cells 7 and 7c
        rows.append({"dataset": label, "spacing x storm size": sp,
                     "worst ring error (m/s)": round(float((ref - w).max()), 2),
                     "Castries error (m/s)": round(at_castries(ref) - at_castries(w), 2),
                     "seconds": round(time.time() - t0, 2)})
spacing_test = pd.DataFrame(rows)
print("Worst ring error (m/s) — smaller is better:")
print(spacing_test.pivot(index="spacing x storm size", columns="dataset", values="worst ring error (m/s)").to_string())
print("\nSeconds per storm:")
print(spacing_test.pivot(index="spacing x storm size", columns="dataset", values="seconds").to_string())
print(f"\nCurrent setting: SPACING = {SPACING}")


CELL 6B: TESTING SPACING VALUES

In [ ]:
cols = TEST_SPACINGS + ["ref"]                                     # coarse -> fine, then the reference (0.05)
field = lambda label, sp: refs[label] if sp == "ref" else spacing_fields[label, sp]
name = lambda sp: "reference\n0.05" if sp == "ref" else f"spacing\n{sp}" + ("  ← used" if sp == SPACING else "")

# 1. Footprints: rows = datasets, columns = spacing (one wind scale for everything)
lim = limits([field(l, sp) for l in catalogues for sp in cols], step=5)
fig, axes = map_figure(len(catalogues), len(cols), extra=1.0, scale=0.55)
for row, label in zip(axes, catalogues):
    for ax, sp in zip(row, cols):
        im = show(ax, field(label, sp), WIND_CMAP, *lim, name(sp))
        ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(f"{label.split(' (')[0]}\n{SHOWCASE[label]}", fontsize=8.5, fontweight="bold")
fig.colorbar(im, ax=axes, shrink=0.5, label="Maximum wind, open terrain (m/s)")
fig.suptitle("Footprint at each eye spacing — rings fade from left to right", fontweight="bold", x=0.01, ha="left")
plt.show()

# 2. Wind missed vs the reference (log scale: 0.1, 1 and 10 m/s evenly spaced)
miss_max = max(float((refs[l] - spacing_fields[l, sp]).max()) for l in catalogues for sp in TEST_SPACINGS)
fig, axes = map_figure(len(catalogues), len(TEST_SPACINGS), extra=1.0, scale=0.55)
for row, label in zip(axes, catalogues):
    for ax, sp in zip(row, TEST_SPACINGS):
        miss = refs[label] - spacing_fields[label, sp]
        im = show(ax, miss.clip(min=0.05), "Purples", 0.05, miss_max, f"spacing {sp}\nworst {float(miss.max()):.2f} m/s",
                  norm="log")
        ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(f"{label.split(' (')[0]}\n{SHOWCASE[label]}", fontsize=8.5, fontweight="bold")
fig.colorbar(im, ax=axes, shrink=0.5, label="Wind missed vs reference (m/s, log scale)")
fig.suptitle("Wind missed by the rings at each spacing", fontweight="bold", x=0.01, ha="left")
plt.show()

# 3. Summary: worst error, Castries error and run time against spacing
# fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
# for label, d in spacing_test.groupby("dataset", sort=False):
#     a1.plot(d["spacing x storm size"], d["worst ring error (m/s)"], "o-", label=label)
#     a2.plot(d["spacing x storm size"], d["Castries error (m/s)"].abs(), "o-")
#     a3.plot(d["spacing x storm size"], d["seconds"], "o-")
# a1.set_yscale("log"); a1.axhline(1, color="0.5", ls=":", lw=0.8); a1.text(0.7, 1.08, "1 m/s", fontsize=7, color="0.4", ha="right")
# for ax, title, unit in ((a1, "Worst ring error anywhere", "m/s (log)"), (a2, "Error at Castries", "m/s"),
#                         (a3, "Run time per storm", "seconds")):
#     ax.axvline(SPACING, color="0.3", ls="--", lw=0.8)
#     ax.invert_xaxis()                                              # coarse on the left, fine on the right
#     ax.set(title=title, xlabel="eye spacing (× storm size)", ylabel=unit)
#     ax.spines[["top", "right"]].set_visible(False)
# a1.legend(fontsize=7, frameon=False)
# fig.suptitle(f"Choosing the spacing — dashed line = setting used ({SPACING})", fontweight="bold", x=0.01, ha="left")
# plt.show()


CELL 7: Coffee Cup Before and After with applying a spacing test

In [ ]:
BEFORE = 0.7                                                   # coarse spacing that shows the rings clearly

def ring_field(label, sp):
    """Open-terrain footprint at spacing sp: reused from Cell 6 if tested there, otherwise calculated."""
    if (label, sp) in spacing_fields:
        return spacing_fields[label, sp]
    return compute_winds(showcase_track(label), grid, interpolation_spacing_factor=sp
                         ).data["max_wind_speed_ms"].isel(event=0)

rings = {label: (ring_field(label, BEFORE), ring_field(label, SPACING)) for label in catalogues}
lim = limits([f for pair in rings.values() for f in pair], step=5)              # one wind scale for all storms
gain_max = max(max(float((after - before).max()) for before, after in rings.values()), 0.5)

fig, axes = map_figure(len(rings), 3, extra=2.0, scale=0.8)
for row, (label, (before, after)) in zip(axes, rings.items()):
    track = showcase_track(label).get_track(SHOWCASE[label])
    show(row[0], before, WIND_CMAP, *lim, f"{label} · {SHOWCASE[label]}\nspacing {BEFORE} (before)")
    im_w = show(row[1], after, WIND_CMAP, *lim, f"\nspacing {SPACING} (after)")
    gain = after - before
    im_d = show(row[2], gain, "Purples", 0, gain_max, f"\nRings removed (up to {float(gain.max()):.1f} m/s)")
    for ax in row[:2]:
        ax.plot(track.lon, track.lat, "-", color="cyan", lw=1.2)
    row[0].text(0.03, 0.03, f"Castries {at_castries(before):.1f} m/s", transform=row[0].transAxes, **NOTE)
    row[1].text(0.03, 0.03, f"Castries {at_castries(after):.1f} m/s", transform=row[1].transAxes, **NOTE)
fig.colorbar(im_w, ax=axes[:, :2], shrink=0.6, label="Maximum wind, open terrain (m/s)")
fig.colorbar(im_d, ax=axes[:, 2], shrink=0.6, label="after minus before (m/s)")
fig.suptitle("Coffee cup rings: a calculation artefact removed by finer eye spacing — one storm per dataset",
             fontweight="bold", x=0.01, ha="left")
plt.show()



CELL 7b:

In [ ]:
# Cell 7b — coffee cup rings vs interpolation RATE (time steps) instead of spacing
TEST_FREQUENCIES = ["3h", "1h", "30min", "15min", "10min", "5min"]
rows, fields = [], {}
for label in catalogues:
    one = showcase_track(label)
    ref = compute_winds(one, grid, interpolation_spacing_factor=0.05).data["max_wind_speed_ms"].isel(event=0)   # "truth"
    for f in TEST_FREQUENCIES:
        t0 = time.time()
        w = compute_winds(one, grid, interpolation_frequency=f,
                          interpolation_spacing_factor=None).data["max_wind_speed_ms"].isel(event=0)
        fields[label, f] = w
        rows.append({"dataset": label, "time step": f,
                     "worst ring error (m/s)": round(float((ref - w).max()), 2),
                     "Castries error (m/s)": round(at_castries(ref) - at_castries(w), 2),
                     "seconds": round(time.time() - t0, 2)})
freq_test = pd.DataFrame(rows)
print("Worst ring error (m/s) by time step — smaller is better:")
print(freq_test.pivot(index="time step", columns="dataset", values="worst ring error (m/s)")
               .reindex(TEST_FREQUENCIES).to_string())
print("\nSeconds per storm:")
print(freq_test.pivot(index="time step", columns="dataset", values="seconds").reindex(TEST_FREQUENCIES).to_string())

# picture: 1 h (malkus default) vs 10 min, for the ring storm
before, after = fields[RING_STORM, "1h"], fields[RING_STORM, "5min"]
lim = limits([before, after], step=5)
fig, axes = map_figure(1, 3, extra=2.0)
show(axes[0, 0], before, WIND_CMAP, *lim, f"{SHOWCASE[RING_STORM]}\nevery 1 hour (default)")
im = show(axes[0, 1], after, WIND_CMAP, *lim, "\nevery 5 minutes")
fig.colorbar(im, ax=axes[0, :2], shrink=0.75, label="Maximum wind, open terrain (m/s)")
im = show(axes[0, 2], after - before, "Purples", 0, max(float((after - before).max()), 0.5), "\nRings removed")
fig.colorbar(im, ax=axes[0, 2], shrink=0.75, label="5 min minus 1 h (m/s)")
fig.suptitle("Coffee cup rings vs time step", fontweight="bold", x=0.01, ha="left")
plt.show()


CELL 7-COMPARE: doing comparison with interpolation frequencies

In [ ]:
cols = TEST_FREQUENCIES + ["ref"]                                  # coarse -> fine, then the reference (spacing 0.05)
field = lambda label, f: refs[label] if f == "ref" else fields[label, f]
name = lambda f: "reference\nspacing 0.05" if f == "ref" else f"every\n{f}" + ("  (default)" if f == "1h" else "")

# 1. Footprints: rows = datasets, columns = time step (one wind scale for everything)
lim = limits([field(l, f) for l in catalogues for f in cols], step=5)
fig, axes = map_figure(len(catalogues), len(cols), extra=1.0, scale=0.5)
for row, label in zip(axes, catalogues):
    for ax, f in zip(row, cols):
        im = show(ax, field(label, f), WIND_CMAP, *lim, name(f))
        ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(f"{label.split(' (')[0]}\n{SHOWCASE[label]}", fontsize=8.5, fontweight="bold")
fig.colorbar(im, ax=axes, shrink=0.5, label="Maximum wind, open terrain (m/s)")
fig.suptitle("Footprint at each time step — rings fade from left to right", fontweight="bold", x=0.01, ha="left")
plt.show()

# 2. Wind missed vs the reference (log scale: 0.1, 1 and 10 m/s evenly spaced)
miss_max = max(float((refs[l] - fields[l, f]).max()) for l in catalogues for f in TEST_FREQUENCIES)
fig, axes = map_figure(len(catalogues), len(TEST_FREQUENCIES), extra=1.0, scale=0.5)
for row, label in zip(axes, catalogues):
    for ax, f in zip(row, TEST_FREQUENCIES):
        miss = refs[label] - fields[label, f]
        im = show(ax, miss.clip(min=0.05), "Purples", 0.05, miss_max, f"every {f}\nworst {float(miss.max()):.2f} m/s",
                  norm="log")
        ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(f"{label.split(' (')[0]}\n{SHOWCASE[label]}", fontsize=8.5, fontweight="bold")
fig.colorbar(im, ax=axes, shrink=0.5, label="Wind missed vs reference (m/s, log scale)")
fig.suptitle("Wind missed by the rings at each time step", fontweight="bold", x=0.01, ha="left")
plt.show()

# 3. Summary: worst error, Castries error and run time against time step (in minutes)
# ft = freq_test.assign(minutes=freq_test["time step"].map(lambda f: pd.Timedelta(f).total_seconds() / 60))
# fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
# for label, d in ft.groupby("dataset", sort=False):
#     line, = a1.plot(d["minutes"], d["worst ring error (m/s)"], "o-", label=label)
#     a1.axhline(spacing_test.query("dataset == @label and `spacing x storm size` == @SPACING")["worst ring error (m/s)"].item(),
#                color=line.get_color(), ls=":", lw=0.9)                # same dataset at spacing 0.2, for comparison
#     a2.plot(d["minutes"], d["Castries error (m/s)"].abs(), "o-")
#     a3.plot(d["minutes"], d["seconds"], "o-")
# a1.set_yscale("log")
# for ax, title, unit in ((a1, f"Worst ring error anywhere\n(dotted = spacing {SPACING})", "m/s (log)"),
#                         (a2, "Error at Castries", "m/s"), (a3, "Run time per storm", "seconds")):
#     ax.axvline(60, color="0.3", ls="--", lw=0.8)                       # malkus default: 1 hour
#     ax.set_xscale("log"); ax.invert_xaxis()                            # coarse on the left, fine on the right
#     ax.set_xticks([180, 60, 30, 15, 10, 5]); ax.set_xticklabels(["3h", "1h", "30m", "15m", "10m", "5m"])
#     ax.minorticks_off()
#     ax.set(title=title, xlabel="time step", ylabel=unit)
#     ax.spines[["top", "right"]].set_visible(False)
# a1.legend(fontsize=7, frameon=False)
# fig.suptitle("Choosing the time step — dashed line = malkus default (1 hour)", fontweight="bold", x=0.01, ha="left")
# plt.show()


CELL 8: Open terrain footrints for the storms

In [ ]:

results = {}
for label, c in catalogues.items():
    t0 = time.time()
    print(f"{label}: {len(c['ts'].track_ids)} storms...", flush=True)
    fp = compute_winds(c["ts"], grid, interpolation_spacing_factor=SPACING, n_workers=N_WORKERS)
    results[label] = dict(ts=c["ts"], n_years=c["n_years"], footprints=fp, wind=fp.data["max_wind_speed_ms"])
    print(f"   done in {(time.time() - t0) / 60:.1f} min", flush=True)


CELL 9: Open terrain footprints plots 

In [ ]:
import hashlib, json
# open-terrain footprints are cached, so re-running is quick; recalculated if storms, spacing or grid change
key = hashlib.md5(json.dumps([list(CATALOGUES), SHOWCASE, SPACING, FINE, VIEW]).encode()).hexdigest()[:10]
CACHE = OUT / "cache" / f"showcase_open_{key}.npz"
CACHE.parent.mkdir(parents=True, exist_ok=True)

if CACHE.exists():
    saved = np.load(CACHE)
    opens = {label: on_grid(saved[f"a{i}"], fine_grid) for i, label in enumerate(catalogues)}
    print("Loaded showcase footprints from cache:", CACHE.name)
else:
    opens = {}
    for label in catalogues:
        t0 = time.time()
        opens[label] = compute_winds(showcase_track(label), fine_grid, interpolation_spacing_factor=SPACING
                                     ).data["max_wind_speed_ms"].isel(event=0)
        print(f"{label}: {time.time() - t0:.1f} s", flush=True)
    np.savez_compressed(CACHE, **{f"a{i}": opens[label].values for i, label in enumerate(catalogues)})

fine_kept = on_grid(roughness(fine_grid), fine_grid)            # share of wind kept, worked out once
showcase = {}
for label in catalogues:
    o = opens[label]
    r = o * fine_kept                                           # same result as downscale_winds
    showcase[label] = dict(id=SHOWCASE[label], track=showcase_track(label).get_track(SHOWCASE[label]),
                           open=o, rough=r, cut=r - o)
foot_lim = limits([s["open"] for s in showcase.values()] + [s["rough"] for s in showcase.values()], step=5)

fig, axes = map_figure(1, len(showcase))
for ax, (label, s) in zip(axes[0], showcase.items()):
    im = show(ax, s["open"], WIND_CMAP, *foot_lim, f"{label}\n{s['id']}")
    hurricane_line(ax, s["open"])
    ax.plot(s["track"].lon, s["track"].lat, "-", color="cyan", lw=1.4)
    ax.text(0.03, 0.03, f"Castries {at_castries(s['open']):.1f} m/s", transform=ax.transAxes, **NOTE)
fig.colorbar(im, ax=axes, shrink=0.75, label="Maximum wind, open terrain (m/s)")
fig.suptitle("Open-terrain footprints — one storm per dataset", fontweight="bold", x=0.01, ha="left")
plt.show()

CELL 10: OPen Terrain Risk Maps

In [ ]:
probs = chance_of_hurricane_force(results)
risk_max = limits(list(probs.values()))[1]
fig, axes = map_figure(1, len(probs))
for ax, (label, p) in zip(axes[0], probs.items()):
    im = show(ax, p, RISK_CMAP, 0, risk_max, label)
    pc = at_castries(p)
    ax.text(0.03, 0.03, f"Castries {pc:.1f}% a year" + (f"\n≈ 1 year in {100 / pc:.0f}" if pc > 0 else ""),
            transform=ax.transAxes, **NOTE)
fig.colorbar(im, ax=axes, shrink=0.75, label=f"Chance per year of wind above {HURRICANE_FORCE:g} m/s (%)")
fig.suptitle("Risk map, open terrain — yearly chance of hurricane-force wind", fontweight="bold", x=0.01, ha="left")
plt.show()


Cell 11: Surface Roughness and its effects

In [ ]:
z0 = on_grid(roughness_from_land_cover(LC, MAP, fine_grid), fine_grid)
kept = on_grid(100 * roughness(fine_grid), fine_grid)
on_land = kept < 94

mapping = pd.read_csv(MAP, comment="#")
z0_of = dict(zip(mapping["glob_cover_2009_id"], mapping["roughness_length_m"]))
with rasterio.open(LC) as src:
    cls = src.read(1, window=from_bounds(*VIEW, src.transform).round_offsets().round_lengths())
    px = src.res[0]
ids, counts = np.unique(cls[~np.isin(cls, WATER_CLASSES)], return_counts=True)
pixel_km2 = (px * 111.32 * np.cos(np.radians(13.9))) * (px * 110.57)     # one land-cover pixel
types = pd.DataFrame({"class": ids, "area_km2": counts * pixel_km2})
types["kept_pct"] = 100 * power_law_scale_factors(types["class"].map(z0_of).to_numpy())
types["name"] = np.where(types["class"].isin(TREE_CLASSES), "Tree cover (by height)",
                         types["class"].map(WC_NAMES).fillna(types["class"].astype(str)))
types["kept_x_area"] = types["kept_pct"] * types["area_km2"]
types = (types.groupby("name", as_index=False)[["area_km2", "kept_x_area"]].sum()
              .assign(kept_pct=lambda d: d["kept_x_area"] / d["area_km2"])
              .drop(columns="kept_x_area").sort_values("kept_pct", ascending=False))

fig = plt.figure(figsize=(MAP_W * 3 + 4.5, MAP_H * 2 + 1.4), layout="constrained")
top, bottom = fig.subfigures(2, 1)
a = top.subplots(1, 3, gridspec_kw=dict(width_ratios=(1, 1, 1.6)))
im = show(a[0], z0, "YlGn", 2e-4, 3.0, "Roughness length z₀ (m)", norm="log")
top.colorbar(im, ax=a[0], shrink=0.8)
im = show(a[1], kept, KEPT_CMAP, *KEPT_RANGE, "Wind kept (%)")
top.colorbar(im, ax=a[1], shrink=0.8)
norm = plt.Normalize(*KEPT_RANGE)
a[2].barh(types["name"], types["area_km2"], color=plt.get_cmap(KEPT_CMAP)(norm(types["kept_pct"])), edgecolor="0.3", lw=0.4)
for y, (area, k) in enumerate(zip(types["area_km2"], types["kept_pct"])):
    a[2].text(area + 4, y, f"{k:.0f}% kept", va="center", fontsize=8)
a[2].set_xlabel("Area on St Lucia (km²)"); a[2].set_title("Land types and the wind each keeps")
a[2].spines[["top", "right"]].set_visible(False); a[2].set_xlim(0, types["area_km2"].max() * 1.3)
top.suptitle("What surface roughness looks like (Global Wind Atlas v4)", fontweight="bold", x=0.01, ha="left")

b = bottom.subplots(1, len(showcase))
cut_min = limits([s["cut"] for s in showcase.values()], step=2)[0]
for ax, (label, s) in zip(b, showcase.items()):
    im = show(ax, s["cut"], CUT_CMAP, cut_min, 0, f"{label}\n{s['id']}")
    ax.text(0.03, 0.03, f"land average {float(s['cut'].where(on_land).mean()):.1f} m/s\n"
                        f"sea average {float(in_view(s['cut'].where(~on_land)).mean()):.1f} m/s",
            transform=ax.transAxes, **NOTE)
bottom.colorbar(im, ax=b, shrink=0.75, label="Change in maximum wind from roughness (m/s)")
bottom.suptitle("How much wind roughness removes from each showcase storm", fontweight="bold", x=0.01, ha="left")
plt.show()
print(types.round(1).to_string(index=False))


Cell 12: roughness on its own

In [ ]:
fig, axes = map_figure(len(showcase), 4, extra=0.4)
for row, (label, s) in zip(axes, showcase.items()):
    kept_storm = (100 * s["rough"] / s["open"].where(s["open"] > 0)).where(on_land)   # sea blanked out

    im_w = show(row[0], s["open"], WIND_CMAP, *foot_lim, f"{label.split()[0]} · {s['id']}\nNo roughness")
    show(row[1], s["rough"], WIND_CMAP, *foot_lim, "\nWith roughness")
    im_d = show(row[2], s["cut"], CUT_CMAP, cut_min, 0, "\nDifference")
    im_k = show(row[3], kept_storm, KEPT_CMAP, *KEPT_RANGE, "\nWind kept (%)")
    row[3].set_facecolor("#e3e8ee")
    for ax, field in ((row[0], s["open"]), (row[1], s["rough"])):
        hurricane_line(ax, field)
        ax.plot(s["track"].lon, s["track"].lat, "-", color="cyan", lw=1.2)

    row[0].text(0.03, 0.03, f"Castries {at_castries(s['open']):.1f} m/s", transform=row[0].transAxes, **NOTE)
    row[1].text(0.03, 0.03, f"Castries {at_castries(s['rough']):.1f} m/s", transform=row[1].transAxes, **NOTE)
    row[2].text(0.03, 0.03, f"Castries {at_castries(s['cut']):.1f} m/s\n"
                            f"land avg {float(s['cut'].where(on_land).mean()):.1f} m/s",
                transform=row[2].transAxes, **NOTE)
    row[3].text(0.03, 0.03, f"land avg {float(kept_storm.mean()):.0f}%", transform=row[3].transAxes, **NOTE)

fig.colorbar(im_w, ax=axes[:, :2], location="bottom", shrink=0.6, label="Maximum wind (m/s)")
fig.colorbar(im_d, ax=axes[:, 2], location="bottom", shrink=0.9, label="Wind removed (m/s)")
fig.colorbar(im_k, ax=axes[:, 3], location="bottom", shrink=0.9, label="Share of wind kept (%)")
fig.suptitle("Surface roughness on its own — before, after, the difference and the share of wind kept",
             fontweight="bold", x=0.01, ha="left")
plt.show()


CELL 13: Footprints with Roughness

In [ ]:
fig, axes = map_figure(1, len(showcase))
for ax, (label, s) in zip(axes[0], showcase.items()):
    im = show(ax, s["rough"], WIND_CMAP, *foot_lim, f"{label}\n{s['id']}")
    hurricane_line(ax, s["rough"])
    ax.plot(s["track"].lon, s["track"].lat, "-", color="cyan", lw=1.4)
    ax.text(0.03, 0.03, f"Castries {at_castries(s['open']):.1f} → {at_castries(s['rough']):.1f} m/s",
            transform=ax.transAxes, **NOTE)
fig.colorbar(im, ax=axes, shrink=0.75, label="Maximum wind, with land roughness (m/s)")
fig.suptitle("Footprints with surface roughness — one storm per dataset", fontweight="bold", x=0.01, ha="left")
plt.show()


CELL 14: Risk Maps with Roughness

In [ ]:
results_rough = {}
for label, r in results.items():
    fp = downscale_winds(r["footprints"], method=roughness)
    results_rough[label] = dict(ts=r["ts"], n_years=r["n_years"], footprints=fp, wind=fp.data["max_wind_speed_ms"])
probs_rough = chance_of_hurricane_force(results_rough)

fig, axes = map_figure(2, len(probs_rough))
change = {label: probs_rough[label] - probs[label] for label in probs_rough}
change_min = limits(list(change.values()))[0]
for col, label in enumerate(probs_rough):
    im_r = show(axes[0, col], probs_rough[label], RISK_CMAP, 0, risk_max, label)
    pc = at_castries(probs_rough[label])
    axes[0, col].text(0.03, 0.03, f"Castries {pc:.1f}% a year" + (f"\n≈ 1 year in {100 / pc:.0f}" if pc > 0 else ""),
                      transform=axes[0, col].transAxes, **NOTE)
    im_c = show(axes[1, col], change[label], CUT_CMAP, change_min, 0, f"{label}\nchange from open terrain")
fig.colorbar(im_r, ax=axes[0], shrink=0.75, label=f"Chance per year of wind above {HURRICANE_FORCE:g} m/s (%)")
fig.colorbar(im_c, ax=axes[1], shrink=0.75, label="Change (percentage points)")
fig.suptitle("Risk map with surface roughness — and how much roughness lowers it", fontweight="bold", x=0.01, ha="left")
plt.show()


CELL 14B: LOOKING AT THE DIFFERENCE IN GRID

In [ ]:
probs_rough_fine = {label: fine_chance_with_roughness(annual_maxima(r["wind"], r["n_years"]), r["n_years"], fine_kept)
                    for label, r in results.items()}

fig, axes = map_figure(1, len(probs_rough_fine))
for ax, (label, p) in zip(axes[0], probs_rough_fine.items()):
    im = show(ax, p, RISK_CMAP, 0, risk_max, label)
    pc = at_castries(p)
    ax.text(0.03, 0.03, f"Castries {pc:.1f}% a year" + (f"\n≈ 1 year in {100 / pc:.0f}" if pc > 0 else ""),
            transform=ax.transAxes, **NOTE)
fig.colorbar(im, ax=axes, shrink=0.75, label=f"Chance per year of wind above {HURRICANE_FORCE:g} m/s (%)")
fig.suptitle("Risk map with surface roughness at 90 m — sharper land detail", fontweight="bold", x=0.01, ha="left")
plt.show()


CELL 15: Return Period Maps

In [ ]:
maps = {label: rp_maps(r["wind"], r["n_years"], RETURN_PERIODS) for label, r in results.items()}      # 1.1 km, open terrain
maps_open_fine = {label: to_fine(m) for label, m in maps.items()}                                     # filled in to 90 m
maps_rough_fine = {label: m * fine_kept for label, m in maps_open_fine.items()}                       # x 90 m share kept
rp_lim = limits(list(maps_open_fine.values()) + list(maps_rough_fine.values()), step=5)

def plot_rp(md, terrain):
    fig, axes = map_figure(len(md), len(RETURN_PERIODS), extra=1.0, scale=0.62)
    for row, (label, m) in zip(axes, md.items()):
        for ax, T in zip(row, RETURN_PERIODS):
            im = show(ax, m.sel(return_period=T), WIND_CMAP, *rp_lim, f"1-in-{T} yr")
            ax.set_xticks([]); ax.set_yticks([])
            if np.isnan(m.sel(return_period=T)).all():                    # longer than the catalogue (e.g. Emanuel)
                ax.text(0.5, 0.5, f"longer than\n{results[label]['n_years']}-yr catalogue",
                        transform=ax.transAxes, ha="center", va="center", fontsize=7.5, color="0.4")
        row[0].set_ylabel(label, fontsize=9, fontweight="bold")
    fig.colorbar(im, ax=axes, shrink=0.6, label=f"Wind speed, {terrain} (m/s)")
    fig.suptitle(f"Return period maps (90 m) — {terrain}", fontweight="bold", x=0.01, ha="left")
    plt.show()

plot_rp(maps_open_fine, "open terrain")
plot_rp(maps_rough_fine, "with land roughness")

rp_table = pd.DataFrame([{"dataset": label, "terrain": terrain,
                          **{f"1-in-{T}": round(at_castries(md[label].sel(return_period=T)), 1) for T in RETURN_PERIODS}}
                         for label in maps for terrain, md in (("open terrain", maps_open_fine),
                                                                ("with roughness (90 m)", maps_rough_fine))])
print("1-in-N-year wind at Castries (m/s)\n" + rp_table.to_string(index=False))


Sharper Risk Maps

In [ ]:
fine_kept = on_grid(roughness(fine_grid), fine_grid)                       # 90 m share of wind kept
fig, axes = map_figure(1, len(maps))
sharp = {label: maps[label].sel(return_period=100).interp(lat=fine_grid.lats, lon=fine_grid.lons) * fine_kept
         for label in maps}                                                # open-terrain map (smooth) x 90 m roughness
lim = limits(list(sharp.values()), step=5)
for ax, (label, m) in zip(axes[0], sharp.items()):
    im = show(ax, m, WIND_CMAP, *lim, f"{label}\n1-in-100-yr, 90 m roughness")
    ax.text(0.03, 0.03, f"Castries {at_castries(m):.1f} m/s", transform=ax.transAxes, **NOTE)
fig.colorbar(im, ax=axes, shrink=0.75, label="1-in-100-year wind, with roughness (m/s)")
plt.show()


RENAMING

In [ ]:
MAP_WORLDCOVER_COPIED.write_text(MAP_WORLDCOVER_COPIED.read_text().replace("land_cover_class_id", "glob_cover_2009_id"))


Cell 16: compare roughness versions (GlobCover vs copied WorldCover table vs GWA4)

In [ ]:

# # versions = {"GlobCover 2009\n(300 m, old table)":   on_grid(SurfaceRoughness(LC_GLOBCOVER, MAP_GLOBCOVER)(fine_grid), fine_grid),
#             "ESA WorldCover\n(10 m, copied table)": on_grid(SurfaceRoughness(LC_WORLDCOVER, MAP_WORLDCOVER_COPIED)(fine_grid), fine_grid),
#             "ESA + ETH = GWA4\n(50 m, DTU table)":  fine_kept}

globcover_grid = RegularGrid.from_bbox(BBOX, 1 / 360)                                  # 1/360° = GlobCover's 300 m pixels
globcover_kept = (on_grid(SurfaceRoughness(LC_GLOBCOVER, MAP_GLOBCOVER)(globcover_grid), globcover_grid)
                  .sel(lat=fine_grid.lats, lon=fine_grid.lons, method="nearest")
                  .assign_coords(lat=fine_grid.lats, lon=fine_grid.lons))

versions = {"GlobCover 2009\n(300 m, old table)":   globcover_kept,
            "ESA WorldCover\n(10 m, copied table)": on_grid(SurfaceRoughness(LC_WORLDCOVER, MAP_WORLDCOVER_COPIED)(fine_grid), fine_grid),
            "ESA + ETH = GWA4\n(50 m, DTU table)":  fine_kept}

# 1. Maps: wind kept in each version, and GWA4 minus the copied WorldCover table
fig, axes = map_figure(1, 4, extra=2.2)
for ax, (name, f) in zip(axes[0, :3], versions.items()):
    im = show(ax, 100 * f, KEPT_CMAP, *KEPT_RANGE, f"Wind kept:\n{name}")
    ax.text(0.03, 0.03, f"land avg {100 * float(f.where(on_land).mean()):.1f}%", transform=ax.transAxes, **NOTE)
fig.colorbar(im, ax=axes[0, :3], shrink=0.75, label="% of open-terrain wind kept")
gwa4_kept, copied_kept = list(versions.values())[2], list(versions.values())[1]
d = 100 * (gwa4_kept - copied_kept)
im = show(axes[0, 3], d, "RdBu_r", -10, 10, "GWA4 minus\ncopied WorldCover table")
axes[0, 3].text(0.03, 0.03, f"land avg {float(d.where(on_land).mean()):+.1f} pts", transform=axes[0, 3].transAxes, **NOTE)
fig.colorbar(im, ax=axes[0, 3], shrink=0.75, label="percentage points (red = GWA4 keeps more wind)")
fig.suptitle("Land cover and roughness tables compared (90 m)", fontweight="bold", x=0.01, ha="left")
plt.show()

# 2. What each version does to the hazard numbers (90 m roughness, same storms)
rows = []
for label, r in results.items():
    n = r["n_years"]
    open_rp100 = maps_open_fine[label].sel(return_period=100)
    curve = chance_above(annual_maxima(r["wind"], n), n).interp(**CASTRIES)        # % of years above each level, at Castries
    for name, f in versions.items():
        rp100 = open_rp100 * f
        chance = float(np.interp(HURRICANE_FORCE / at_castries(f), CHANCE_LEVELS, curve.values))   # same method as Cell 14b
        rows.append({"dataset": label, "roughness version": name.replace("\n", " "),
                     "Castries 1-in-100-yr (m/s)": round(at_castries(rp100), 1),
                     "island mean 1-in-100-yr (m/s)": round(float(rp100.where(on_land).mean()), 1),
                     "Castries % chance/yr": round(chance, 1)})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
print("\nIsland-average wind kept: " + "   ".join(f"{name.splitlines()[0]} {100 * float(f.where(on_land).mean()):.1f}%"
                                               for name, f in versions.items()))
# 3. The effect on the hazard: 1-in-100-yr wind under each land-cover version (all storms in each catalogue)
fig, axes = map_figure(len(results), len(versions), extra=1.2, scale=0.7)
for row, label in zip(axes, results):
    open_rp100 = maps_open_fine[label].sel(return_period=100)
    for ax, (name, f) in zip(row, versions.items()):
        im = show(ax, open_rp100 * f, WIND_CMAP, *rp_lim, name.splitlines()[0] if row is axes[0] else "")
        ax.text(0.03, 0.03, f"Castries {at_castries(open_rp100 * f):.1f} m/s", transform=ax.transAxes, **NOTE)
        ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(f"{label}\nall storms", fontsize=8.5, fontweight="bold")
fig.colorbar(im, ax=axes, shrink=0.6, label="1-in-100-year wind, with roughness (m/s)")
fig.suptitle("1-in-100-year wind under each land-cover version — rows: datasets (all storms), columns: land cover",
             fontweight="bold", x=0.01, ha="left")
plt.show()

# 4. Castries summary: one bar per land-cover version, grouped by dataset
pivot = comparison.pivot(index="dataset", columns="roughness version", values="Castries 1-in-100-yr (m/s)").loc[list(results)]
ax = pivot.plot.bar(figsize=(9, 3.8), rot=0, color=["#bdbdbd", "#6baed6", "#238b45"], edgecolor="0.3", lw=0.4)
for c in ax.containers:
    ax.bar_label(c, fmt="%.1f", fontsize=7)
ax.set(ylabel="1-in-100-yr wind at Castries (m/s)", xlabel="",
       title="Castries 1-in-100-year wind: how much the land-cover choice matters")
ax.legend(fontsize=7.5, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
ax.spines[["top", "right"]].set_visible(False)
plt.show()
